# TorchRL DQN

In this tutorial, we will learn how to use DQN!

* Note: this notebook was modified from the official tutorials provided by torchRL, you can check it out here: https://docs.pytorch.org/rl/stable/tutorials

### Install TorchRL

First let's install torchrl and tensordict.


In [ ]:
#Option 1: easy
#If you are running this from colab you can run:
#!pip install tensordict
#!pip install torchrl

#Option 2: harder
#if you are using your own compputer you should create a virtual environment, activate it, and then use pip to install the needed 
#conda create -n torchrl-env python=3.11 -y
#conda activate torchrl-env
#pip install torchrl tensordict torch torchvision 'gymnasium[all]'
#Note: for GPU support this might be a little different, and might vary by device!

Load some required libraries

In [ ]:
from tensordict.nn import TensorDictSequential
from torchrl.modules import EGreedyModule

import torch
from tensordict.nn import TensorDictModule
from tensordict import TensorDict
from torchrl.envs import GymEnv
from torch import nn

from torchrl.collectors import SyncDataCollector
from torchrl.collectors import RandomPolicy

from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.modules import QValueActor
from torch.optim import Adam
from torchrl.modules import QValueActor
from torchrl.objectives import DQNLoss

from torchrl.envs.transforms import RewardSum
from torchrl.envs import TransformedEnv, Compose, StepCounter


### Data collectors

In the last tutorial we started to allow our agents to learn. We learned how to implemented a DQN algorithm and how it works internally. 

But we haven't shown how we can do this repeately through time, with many agent-enviornment interactions. Let's learn how to collect and use this data, and handle interactions between agents and their environment.

To do this we will learn about *DataCollectors*

The primary data collector discussed here is the SyncDataCollector, which is the focus of this documentation. At a fundamental level, a collector is a straightforward class responsible for executing your policy within the environment, resetting the environment when necessary, and providing batches of data of a predefined size. Unlike the rollout() method demonstrated in the earlier tutorials, collectors do not reset between consecutive batches of data. Consequently, two successive batches of data may contain elements from the same trajectory.

The basic arguments you need to pass to your collector are:

* the size of the batches you want to collect (frames_per_batch), 
* the length (possibly infinite) of the iterator, 
* the policy and the environment. 

For simplicity, we will use a dummy, random policy in this first example.

In [ ]:
torch.manual_seed(0)

env = GymEnv("CartPole-v1")
env.set_seed(0)

policy = RandomPolicy(env.action_spec)
collector = SyncDataCollector(env, policy, frames_per_batch=200, total_frames=-1)

We now expect that our collector will deliver batches of size 200 no matter what happens during collection. In other words, we may have multiple trajectories in this batch! The total_frames indicates how long the collector should be. A value of -1 will produce a never ending collector.

Let’s iterate over the collector to get a sense of what this data looks like:

In [ ]:
for data in collector:
    print(data)
    break

As you can see, our data is augmented with some collector-specific metadata grouped in a "collector" sub-tensordict that we did not see during environment rollouts. This is useful to keep track of the trajectory ids. In the following list, each item marks the trajectory number the corresponding transition belongs to:

In [ ]:
print(data["collector", "traj_ids"])


Because we are using the cartpole environment, each time the pole falls the environment is reset and we get another trajectory ID.

Data collectors are very useful when it comes to coding state-of-the-art algorithms, as performance is usually measured by the capability of a specific technique to solve a problem in a given number of interactions with the environment (the total_frames argument in the collector). For this reason, most training loops in our examples look like this:

In [ ]:
#for data in collector:
#     your algorithm here

### Replay buffers

Now that we have explored how to collect data, we would like to know how to store it. In RL, the typical setting is that the data is collected, stored temporarily and cleared after a little while given some heuristic: first-in first-out or random. 

A typical pseudo-code would look like this:

In [ ]:

#for data in collector:
#     storage.store(data)
#     for i in range(n_optim):
#         sample = storage.sample()
#         loss_val = loss_fn(sample)
#         loss_val.backward()
#         optim.step() 


The parent class that stores the data in TorchRL is referred to as ReplayBuffer. TorchRL’s replay buffers are composable: you can edit the storage type, their sampling technique, the writing heuristic or the transforms applied to them. We will leave the fancy stuff for a dedicated in-depth tutorial. The generic replay buffer only needs to know what storage it has to use. In general, we recommend a TensorStorage subclass, which will work fine in most cases. We’ll be using LazyMemmapStorage in this tutorial, which enjoys two nice properties: first, being “lazy”, you don’t need to explicitly tell it what your data looks like in advance. Second, it uses MemoryMappedTensor as a backend to save your data on disk in an efficient way. The only thing you need to know is how big you want your buffer to be.

In [ ]:
from torchrl.data.replay_buffers import LazyMemmapStorage, ReplayBuffer
import tempfile

buffer_scratch_dir = tempfile.TemporaryDirectory().name

buffer = ReplayBuffer(
    storage=LazyMemmapStorage(max_size=1000, scratch_dir=buffer_scratch_dir)
)


Or for smaller replay buffers, like the ones we'll work with today, we can keep everything in memory.

In [ ]:
from torchrl.data import LazyTensorStorage

buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=1000)
)

Populating the buffer can be done via the add() (single element) or extend() (multiple elements) methods. Using the data we just collected, we initialize and populate the buffer in one go:

In [ ]:
indices = buffer.extend(data)

We can check that the buffer now has the same number of elements as what we got from the collector:



In [ ]:
assert len(buffer) == collector.frames_per_batch


The only thing left to know is how to gather data from the buffer. Naturally, this relies on the sample() method. Because we did not specify that sampling had to be done without repetitions, it is not guaranteed that the samples gathered from our buffer will be unique:

In [ ]:
sample = buffer.sample(batch_size=30)
print(sample)

### DQN

Let's put this all together and try the collector and replay buffer out with the DQN algorithm we learnt in the last tutorial.

Let's define our policy 

In [ ]:
#create the environment
env = GymEnv("CartPole-v1", from_pixels=True, pixels_only=False)

#add in step counter and reward sums per episode
env = TransformedEnv(env, Compose(
        StepCounter(),
        RewardSum()
        )
    )

n_obs = env.observation_spec["observation"].shape[-1]
n_act = env.action_spec.shape[-1]
nodes = 128

#create a model: takes observations as inputs, and outputs action values
model = nn.Sequential(
    nn.Linear(n_obs,nodes),
    nn.ReLU(),
    nn.Linear(nodes,nodes),
    nn.ReLU(),
    nn.Linear(nodes, n_act),
)

#go from observations to action values using the model
value_net = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["action_value"],          
)

#let's use the qvalueactor to help choose actions
qvalue_actor = QValueActor(module=value_net, spec=env.action_spec)

#when collecting data sometimes use random (e-greedy)
exploration_module = EGreedyModule(
    spec=env.action_spec, 
    annealing_num_steps=50000, 
    eps_init=0.95,
    eps_end=0.05
)
qvalue_actor_explore = TensorDictSequential(qvalue_actor, exploration_module)


Let's then define a loss function and an optimizer


In [ ]:
dqn_loss = DQNLoss(qvalue_actor) #when updating use a greedy strategy (always choose best action)

optim = Adam(dqn_loss.parameters(), lr=0.0001, weight_decay=1e-4)

Now let's write a loop that uses the collector and uses batches of data from a rollout.

* Note: this loop might take around 5-10min to run! 

Let's take note of how much new experiences we are adding to the replay buffer (data generation). We control this by setting the frames_per_batch. Right now it is 200, so every 200 steps we will add these new data to the replay buffer.

Let's also take note of how much we are consuming data during updates. We can control this by changing the number of updates to perform for each new piece of data added to the replay buffer. We can measure this by N_updates / frames_per_batch. When this ratio is high we get high sample efficiency, but it comes at the cost of potential instability and overfitting. When this ratio is low we get low sample efficiency, and computation can take a lot longer.

In [ ]:

total_frames = 200000 # Total number of steps the agent will take in it's environment
frames_per_batch = 200 # how many steps the agent will take before we add this data to the replay buffer
n_updates = 5 # for each new addition to the replay buffer, we will perform this many updates to the policy
buffer_size = 5000 # how many steps to store. If this is large it will store more past agent-env interactions, if small, it will store only the most recent interactions.
min_buffer_size = 1000 # minimum size of the buffer before training (at the start the buffer must fill up)

print(f"UTD ratio: {n_updates/frames_per_batch}")
print_every = 10

#create the collector
collector = SyncDataCollector(env, qvalue_actor_explore, frames_per_batch=frames_per_batch, total_frames=total_frames)

#create the replay buffer
buffer_dqn = ReplayBuffer(
    storage=LazyTensorStorage(max_size=buffer_size)
)

#list to keep track of loss over time (we should see this increase if the agent is getting better at predicting the value of actions in states)
loss_over_time = []
episode_rewards = []

#agent-environment interactions!
for data in collector:
    
    #update buffer with new data
    buffer_dqn.extend(data)

    #update the e-greedy module
    exploration_module.step(frames_per_batch)

    if(len(buffer_dqn)<min_buffer_size):
        continue

    for _ in range(n_updates):

        #sample from the buffer
        sample = buffer_dqn.sample(batch_size=64)

        #calculate the loss
        loss = dqn_loss(sample)
        loss_val = loss["loss"]

        #let's now optimize the model used to estimate q-values
        optim.zero_grad(set_to_none=True)
        loss_val.backward() #backprop!
        optim.step()

    #let's keep track of the loss over time
    loss_over_time.append(loss_val.item())

    # log episode returns
    done = data["next", "done"].squeeze()
    if done.any():
        for ep_reward in data["next", "episode_reward"][done]:
            episode_rewards.append(ep_reward.item())

            if len(episode_rewards) % print_every == 0:
                recent_mean = sum(episode_rewards[-print_every:]) / print_every
                print(f"Episode {len(episode_rewards)} | "
                    f"last {print_every} mean: {recent_mean:.1f} | "
                    f"last reward: {episode_rewards[-1]:.1f} | "
                    f"loss: {loss_val:.3f} | "
                    f"eps: {exploration_module.eps.item():.3f}"
                    )


Let's take a look at how the loss changed over time. I.e., how well is the agent predicting the TD target?

In [ ]:
import matplotlib.pyplot as plt

plt.plot(loss_over_time, alpha=0.3, label="raw")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("DQN Loss over Time")
plt.legend()
plt.show()

Ideally, you should see a drop in loss over time. Remember this means that the agent is getting better at predicting the TD target (i.e., the reward it will recieve plus the discounted estimate of the value of the next state-action). So it's learning how it's actions lead to rewards and more valuable states.

Likely what you might be seeing is an initial drop, then lots of variability. Where in some cases the loss explodes and then comes back down.

Let's look at how the rewards collected within episodes changes over time as well.

In [ ]:
import pandas as pd

smoothed = pd.Series(episode_rewards).rolling(window=10).mean()

plt.plot(episode_rewards, alpha=0.3, label="raw")

plt.plot(smoothed, label="smoothed (50-episode avg)")
plt.xlabel("Episode")
plt.ylabel("Episode Reward")
plt.title("DQN Episode Reward over Time")
plt.legend()
plt.show()

Here you are likely seeing that the agent is starting to learn a policy that results in rewards increasing over time. We might have to train a little longer...

We can likely run this longer and eventually the agent will figure this out. Or we can add what's known as a target network to help with training stability.

### Target network

You'll notice that in the last loop we got a warning related to a target network. In supervised learning tasks the outcome you are trying to predict doesn't change as you optimize your model. In RL that's not the case. For example, in DQN, we are using our model to make predictions about the value of actions in states, and we using that same model to generate agent-environment data about values of actions within states. That means we are trying to change the weights of our model to make predictions about a target (value of actions in states) that is always somewhat moving. This can make learning hard. 

In DQN it's been found that if we have a copy of our current network that is used to predict the targets, and slowly update it, we can stabilize learning. That is we'll have a network that is used to take actions (online network) and a copy of that network that will be used for predicting the value of actions in states (target network).

Let's use a soft update approach, where we slowly update the target network.

To do so let's first setup our qvalue-actor:

In [ ]:
from torchrl.objectives import SoftUpdate
from torchrl.envs import ObservationNorm


env = GymEnv("CartPole-v1")

#add in step counter and reward sums per episode
env = TransformedEnv(env, Compose(
        StepCounter(),
        RewardSum(),
        ObservationNorm(in_keys=["observation"], standard_normal=True)
        )
    )
env.transform[-1].init_stats(num_iter=1000, reduce_dim=0, cat_dim=0)


n_obs = env.observation_spec["observation"].shape[-1]
n_act = env.action_spec.space.n

nodes = 128

#create a model: takes observations as inputs, and outputs action values
model = nn.Sequential(
    nn.Linear(n_obs,nodes),
    nn.ReLU(),
    nn.Linear(nodes,nodes),
    nn.ReLU(),
    nn.Linear(nodes, n_act),
)

#go from observations to action values using the model
value_net = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["action_value"],          
)

#let's use the qvalueactor to help choose actions
qvalue_actor = QValueActor(module=value_net, spec=env.action_spec)

#when collecting data sometimes use random (e-greedy)
exploration_module = EGreedyModule(
    spec=env.action_spec, 
    annealing_num_steps=80000, #should be chosen relative to total_frames below 
    eps_init=0.95,
    eps_end=0.05
)
qvalue_actor_explore = TensorDictSequential(qvalue_actor, exploration_module)




Then let's setup the DQNLoss. Here we will specify that we'd like to use double DQN (i.e., have two networks) with a delay in updating the target network.

* double_dqn: whether to use two networks
* delay_value: whether to update the target based on the online network 

We can also make sure to specify a discount factor explicitly (gamma=0.99), and define how slowly to update the target network (tau=0.001)

In [ ]:
from torchrl.objectives import SoftUpdate

dqn_loss = DQNLoss(
    value_network=qvalue_actor,
    action_space=env.action_spec,
    delay_value=True,          # create a target network (slowly updates)
    double_dqn=True,
)
dqn_loss.make_value_estimator(gamma=0.99)
updater = SoftUpdate(dqn_loss, tau=0.001)  # small tau = slow target update

#create an optimizer
optim = Adam(dqn_loss.parameters(), lr=0.0001, weight_decay=1e-4)


Let's write out the same loop as above, with some additions:

* We'll now perform a soft update to the target network within the loop.
* We'll also add some code to save the best model, so we can use it later on.

In [ ]:

total_frames = 400000 # Total number of steps the agent will take in it's environment
frames_per_batch = 200 # how many steps the agent will take before we add this data to the replay buffer
n_updates = 5 # for each new addition to the replay buffer, we will perform this many updates to the policy
buffer_size = 5000 # how many steps to store. If this is large it will store more past agent-env interactions, if small, it will store only the most recent interactions.
min_buffer_size = 1000 # only start training when the buffer has enough data (good for the start)

print(f"UTD ratio: {n_updates/frames_per_batch}")
print_every=10

collector = SyncDataCollector(env, qvalue_actor_explore, frames_per_batch=frames_per_batch, total_frames=total_frames)

#create the replay buffer
buffer_dqn = ReplayBuffer(
    storage=LazyTensorStorage(max_size=buffer_size)
)


loss_over_time = []
episode_rewards = []
best_reward = 0


#agent-environment interactions!
for data in collector:
    
    #add data to the buffer
    buffer_dqn.extend(data)

    #update the e-greedy module
    exploration_module.step(frames_per_batch)


    # check length of buffer
    if len(buffer_dqn) < min_buffer_size:
        continue

    for _ in range(n_updates):
    
        #add new experiences to the buffer
        sample = buffer_dqn.sample(batch_size=64)

        #calculate the loss
        loss = dqn_loss(sample)
        loss_val = loss["loss"]
        
        #let's now optimize the model used to estimate q-values
        optim.zero_grad(set_to_none=True) #make sure everything gets set to zero before backprob..
        loss_val.backward() #backprop!
        optim.step()
        updater.step() #now updates the target network as well
    
    #let's keep track of the loss over time
    loss_over_time.append(loss_val.item())

    # log episode returns
    done = data["next", "done"].squeeze()
    if done.any():
        for ep_reward in data["next", "episode_reward"][done]:
            episode_rewards.append(ep_reward.item())

            # save best model
            if ep_reward.item() > best_reward:
                best_reward = ep_reward.item()
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'obs_norm_loc': env.transform[-1].loc,
                    'obs_norm_scale': env.transform[-1].scale,
                }, "best_model.pt")
            
            # print some diagnostics while training
            if len(episode_rewards) % print_every == 0:
                recent_mean = sum(episode_rewards[-print_every:]) / print_every
                print(f"Episode {len(episode_rewards)} | "
                    f"last {print_every} mean: {recent_mean:.1f} | "
                    f"last reward: {episode_rewards[-1]:.1f} | "
                    f"loss: {loss_val:.3f} | "
                    f"eps: {exploration_module.eps.item():.3f}"
                    )



In [ ]:
import pandas as pd
#import matplotlib.pyplot as plt
smoothed = pd.Series(loss_over_time).rolling(window=50).mean()

plt.plot(loss_over_time, alpha=0.3, label="raw")
plt.plot(smoothed, label="smoothed (50-step avg)")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("DQN Loss over Time")
plt.legend()
plt.show()

You should see that the addition of the target network along with the policy network helps with calculating the loss. (not yet!)



In [ ]:

smoothed = pd.Series(episode_rewards).rolling(window=50).mean()

plt.plot(episode_rewards, alpha=0.3, label="raw")
plt.plot(smoothed, label="smoothed (50-step avg)")
plt.xlabel("Training Step")
plt.ylabel("Rewards")
plt.title("Rewards Time")
plt.legend()
plt.show()

As a last step, let's visualize your trained agent in it's environment!

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import imageio
from IPython.display import Image
from torchrl.envs import TransformedEnv
from torchrl.envs.transforms import Compose, StepCounter, RewardSum

# load checkpoint
checkpoint = torch.load("best_model.pt")

# load model weights
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# recreate environment with saved normalisation stats
env_render = GymEnv("CartPole-v1", from_pixels=True, pixels_only=False)

obs_norm = ObservationNorm(in_keys=["observation"], standard_normal=True)

env_render = TransformedEnv(env_render, Compose(
    StepCounter(),
    RewardSum(),
    obs_norm
))

# manually set the saved stats instead of re-initialising
env_render.transform[-1].loc = checkpoint['obs_norm_loc']
env_render.transform[-1].scale = checkpoint['obs_norm_scale']

# rollout
with torch.no_grad():
    rollout = env_render.rollout(
        max_steps=500,
        policy=qvalue_actor,
        break_when_any_done=True
    )

frames = rollout["pixels"].numpy()
imageio.mimsave("best_policy.gif", frames, fps=30)
Image("best_policy.gif")

**things to try**

* Make a list of all the hyperparameters we've used so far.
* Try changing some of the hyperparamters to see when the training works, or doesn't work! 
> * Feel free to work in groups to better explore parameter space together.
> * Try changing the e-greedy module parameters. Does the agent explore enough, or does it fixate at a suboptimal policy too early.
> * Try changing the learning rate of the online network (lr in optim) vs the learning rate of the target network (soft update). Does it work better when one is more stable (slower learning) than another?
> * Try changing the replay buffer size. Is it better to have a large buffer with lots of experience to learn from, or a small buffer with only the most recent experiences to learn from.
>* DQN should be able to solve this environment, and keep that pole upright indefinitely.
 
* Try DQN with another environment with discrete action spaces.
> * https://gymnasium.farama.org/environments/classic_control/
> * https://gymnasium.farama.org/environments/toy_text/

### Next Steps

Now we know how to use data collectors and replay buffers to feed our algorithm agent-environment inteactions! 

Let's see some better ways to log and visualize our progress!

In the meantime feel free to:
* Learn more about [replay buffers](https://docs.pytorch.org/rl/stable/tutorials/rb_tutorial.html#rb-tuto)
* Learn more about [collectors](https://docs.pytorch.org/rl/stable/reference/collectors_basics.html#ref-collectors) 